## AI 数学基础

### KL散度与交叉熵：
1. **KL 散度**（Kullback-Leibler Divergence）：
   $$
   D_{KL}(P || Q) = \sum_{i} P(i) \log \frac{P(i)}{Q(i)}
   $$
   $$
   D_{KL}(Q || P) = \sum_{i} Q(i) \log \frac{Q(i)}{P(i)}
   $$

2. **交叉熵**（Cross Entropy）：
   $$
   H(Q, P) = -\sum_{i} Q(i) \log P(i)
   $$


In [10]:
import numpy as np

# 定义两个离散分布 p 和 q
p = np.array([0.2, 0.3, 0.4, 0.1])
q = np.array([0.1, 0.2, 0.4, 0.3])

# 计算 KL 散度 D_KL(q || p)
D_KL_q_p = np.sum(q * np.log(q / p))

# 计算 KL 散度 D_KL(p || q)
D_KL_p_q = np.sum(p * np.log(p / q))

# 计算 CrossEntropy(q, p)
cross_entropy_q_p = -np.sum(q * np.log(p))

print("KL 散度 D_KL(q || p)：", D_KL_q_p)
print("KL 散度 D_KL(p || q)：", D_KL_p_q)
print("CrossEntropy(q, p)：", cross_entropy_q_p)

KL 散度 D_KL(q || p)： 0.17917594692280547
KL 散度 D_KL(p || q)： 0.15040773967762736
CrossEntropy(q, p)： 1.459030172756473


### 反向传播
假设一个三层神经网络（两个隐藏层）有以下结构：
- 输入层：特征维度为 $d$ ，输入为 $x \in \mathbb{R}^d$
- 第一隐藏层：有 $h_1$ 个神经元，激活函数为 $\sigma$ ，权重矩阵为 $W_1 \in \mathbb{R}^{h_1 \times d}$
- 第二隐藏层：有 $h_2$ 个神经元，激活函数为 $\sigma$ ，权重矩阵为 $W_2 \in \mathbb{R}^{h_2 \times h_1}$
- 输出层：有 $d^{\prime}$ 个神经元，无激活函数，权重矩阵为 $W_3 \in \mathbb{R}^{d^{\prime} \times h_2}$

第一隐藏层的輸出：
第二隐藏层的输出：

$$
\begin{array}{ll}
z_1=W_1 x \in \mathbb{R}^{h_1}, & a_1=\sigma\left(z_1\right) \in \mathbb{R}^{h_1} \\
z_2=W_2 a_1 \in \mathbb{R}^{h_2}, & a_2=\sigma\left(z_2\right) \in \mathbb{R}^{h_2}
\end{array}
$$


输出层的输出：
为了简单，考虑如下损失函数：

$$
\hat{y}=z_3=W_3 a_2 \in \mathbb{R}^{d^{\prime}}
$$


$$
L=\frac{1}{2}\|\widehat{y}-y\|_2^2
$$

**输出层的梯度计算：**

损失函数对输出层的输出 $z_3$ 的梯度，$z_3 \in \mathbb{R}^{d^{\prime}}$ ：

$$
\frac{\partial L}{\partial z_3}=\hat{y}-\mathrm{y} \in \mathbb{R}^{d^{\prime}}
$$


损失函数对 $W_3$ 的梯度（链式法则），$W_3 \in \mathbb{R}^{d^{\prime} \times h_2}$ ：

$$
\frac{\partial L}{\partial W_3}=\frac{\partial L}{\partial z_3} \frac{\partial z_3}{\partial W_3}=(\hat{y}-\mathrm{y}) a_2^{\top} \in \mathbb{R}^{d^{\prime} \times h_2}
$$


损失函数对 $a_2$ 的梯度，$a_2 \in \mathbb{R}^{h_2}$ ：

$$
\frac{\partial L}{\partial a_2}=\frac{\partial L}{\partial z_3} \frac{\partial z_3}{\partial a_2}=W_3^{\top}(\hat{y}-\mathrm{y}) \in \mathbb{R}^{h_2}
$$

**第二隐藏层的梯度计算：**

损失函数对 $z_2$ 的梯度，$z_2 \in \mathbb{R}^{h_2}$ ：

$$
\frac{\partial L}{\partial z_2}=\frac{\partial L}{\partial a_2} \frac{\partial a_2}{\partial z_2}=W_3^{\top}(\hat{y}-\mathrm{y}) \odot \sigma^{\prime}\left(z_2\right) \in \mathbb{R}^{h_2}
$$


其中 $\odot$ 表示两个对象逐元素相乘
损失函数对 $W_2$ 的梯度，$W_2 \in \mathbb{R}^{h_2 \times h_1}$ ：

$$
\frac{\partial L}{\partial W_2}=\frac{\partial L}{\partial z_2} \frac{\partial z_2}{\partial W_2}=\left[W_3^{\top}(\hat{y}-y) \odot \sigma^{\prime}\left(z_2\right)\right] a_1^{\top} \in \mathbb{R}^{h_2 \times h_1}
$$


损失函数对 $a_1$ 的梯度，$a_2 \in \mathbb{R}^{h_1}$ ：

$$
\frac{\partial L}{\partial a_1}=\frac{\partial L}{\partial z_2} \frac{\partial z_2}{\partial a_1}=W_2^{\top}\left[W_3^{\top}(\hat{y}-y) \odot \sigma^{\prime}\left(z_2\right)\right] \in \mathbb{R}^{h_1}
$$

**第一隐藏层的梯度计算：**

损失函数对 $z_1$ 的梯度，$z_1 \in \mathbb{R}^{h_1}$ ：

$$
\frac{\partial L}{\partial z_1}=\frac{\partial L}{\partial a_1} \frac{\partial a_1}{\partial z_1}=\left[W_2^{\top}\left[W_3^{\top}(\hat{y}-y) \odot \sigma^{\prime}\left(z_2\right)\right]\right] \odot \sigma^{\prime}\left(z_1\right) \in \mathbb{R}^{h_1}
$$


其中 $\odot$ 表示两个对象逐元素相乘
损失函数对 $W_1$ 的梯度，$W_1 \in \mathbb{R}^{h_1 \times d}$ ：

$$
\frac{\partial L}{\partial W_1}=\frac{\partial L}{\partial z_1} \frac{\partial z_1}{\partial W_1}=\left[\left[W_2^{\top}\left[W_3^{\top}(\hat{y}-y) \odot \sigma^{\prime}\left(z_2\right)\right]\right] \odot \sigma^{\prime}\left(z_1\right)\right] x^{\top} \in \mathbb{R}^{h_1 \times d}
$$

In [11]:
import torch

# 定义激活函数及其导数
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

def sigmoid_derivative(x):
    sig = sigmoid(x)
    return sig * (1 - sig)

# 初始化参数
torch.manual_seed(42)  # 设置随机种子保证可复现

d = 4  # 输入维度
h1 = 5  # 第一隐藏层神经元数
h2 = 3  # 第二隐藏层神经元数
d_out = 2  # 输出层神经元数

# 随机初始化权重
W1 = torch.randn(h1, d, requires_grad=True)
W2 = torch.randn(h2, h1, requires_grad=True)
W3 = torch.randn(d_out, h2, requires_grad=True)

# 随机初始化输入和真实标签
x = torch.randn(d, requires_grad=False)
y = torch.randn(d_out, requires_grad=False)

# 前向传播计算
z1 = W1 @ x
a1 = sigmoid(z1)

z2 = W2 @ a1
a2 = sigmoid(z2)

z3 = W3 @ a2
y_hat = z3  # 无激活函数

# 计算损失
loss = 0.5 * torch.norm(y_hat - y) ** 2

In [12]:
# 计算输出层梯度
dL_dz3 = y_hat - y  # ∂L/∂z3
dL_dW3 = dL_dz3.view(-1, 1) @ a2.view(1, -1)  # ∂L/∂W3
dL_da2 = W3.T @ dL_dz3  # ∂L/∂a2

# 计算第二隐藏层梯度
dL_dz2 = dL_da2 * sigmoid_derivative(z2)  # ∂L/∂z2
dL_dW2 = dL_dz2.view(-1, 1) @ a1.view(1, -1)  # ∂L/∂W2
dL_da1 = W2.T @ dL_dz2  # ∂L/∂a1

# 计算第一隐藏层梯度
dL_dz1 = dL_da1 * sigmoid_derivative(z1)  # ∂L/∂z1
dL_dW1 = dL_dz1.view(-1, 1) @ x.view(1, -1)  # ∂L/∂W1

In [13]:
# 计算 PyTorch 自动梯度
# 清除梯度
W1.grad = None
W2.grad = None
W3.grad = None
loss.backward()

# 获取 PyTorch 计算的梯度
grad_W1_torch = W1.grad.clone()
grad_W2_torch = W2.grad.clone()
grad_W3_torch = W3.grad.clone()

# 比较手动计算的梯度和 PyTorch 计算的梯度
print("手动计算的 ∂L/∂W1：\n", dL_dW1)
print("PyTorch 计算的 ∂L/∂W1：\n", grad_W1_torch)

print("\n手动计算的 ∂L/∂W2：\n", dL_dW2)
print("PyTorch 计算的 ∂L/∂W2：\n", grad_W2_torch)

print("\n手动计算的 ∂L/∂W3：\n", dL_dW3)
print("PyTorch 计算的 ∂L/∂W3：\n", grad_W3_torch)

手动计算的 ∂L/∂W1：
 tensor([[ 0.0091,  0.0043, -0.0019, -0.0068],
        [ 0.0800,  0.0378, -0.0165, -0.0598],
        [-0.1103, -0.0521,  0.0228,  0.0824],
        [ 0.0613,  0.0290, -0.0127, -0.0459],
        [ 0.0226,  0.0107, -0.0047, -0.0169]], grad_fn=<MmBackward0>)
PyTorch 计算的 ∂L/∂W1：
 tensor([[ 0.0091,  0.0043, -0.0019, -0.0068],
        [ 0.0800,  0.0378, -0.0165, -0.0598],
        [-0.1103, -0.0521,  0.0228,  0.0824],
        [ 0.0613,  0.0290, -0.0127, -0.0459],
        [ 0.0226,  0.0107, -0.0047, -0.0169]])

手动计算的 ∂L/∂W2：
 tensor([[-0.3343, -0.0546, -0.1663, -0.1059, -0.1536],
        [ 0.3383,  0.0552,  0.1683,  0.1072,  0.1555],
        [ 0.4611,  0.0753,  0.2294,  0.1461,  0.2119]], grad_fn=<MmBackward0>)
PyTorch 计算的 ∂L/∂W2：
 tensor([[-0.3343, -0.0546, -0.1663, -0.1059, -0.1536],
        [ 0.3383,  0.0552,  0.1683,  0.1072,  0.1555],
        [ 0.4611,  0.0753,  0.2294,  0.1461,  0.2119]])

手动计算的 ∂L/∂W3：
 tensor([[-0.5561, -0.8548, -0.6943],
        [-1.0307, -1.5843, -1.2869